# IaC Generation Benchmark Analysis
This notebook analyzes the multi-agent results of running an Infrastructure as Code (IaC) benchmark.
It includes detailed analytics on stage pass rates, terminal failures, and specific error analyses for deployment and validations.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import os

# Set plot style
plt.style.use('ggplot')
sns.set_palette('Set2')

## 1. Load Data
Define the mapping of models to their respective benchmark aggregated results (`.csv` files).

In [ ]:
# Map model names to their CSV file paths
model_files = {
    'Grok Deployable': 'grok_deployable_results_agg.csv',
    'Grok Secure': 'grok_secure_results_agg.csv'
}

dataframes = {}
for model, path in model_files.items():
    if os.path.exists(path):
        dataframes[model] = pd.read_csv(path)
    else:
        print(f"Warning: {path} not found.")
        
print(f"Successfully loaded data for models: {list(dataframes.keys())}")

## 2. Stage Pass Rate Analysis
Analyzing the pass rates across different validation stages: YAML, CFN-Lint, Security (e.g., Trivy), and Deployability.

In [ ]:
stage_stats = []
for model, df in dataframes.items():
    total = len(df)
    if total == 0: continue
    
    # YAML
    if 'val_stage_yaml_passed' in df.columns:
        stage_stats.append({'Model': model, 'Stage': 'YAML', 'Pass Rate (%)': df['val_stage_yaml_passed'].mean() * 100})
    # CFN-Lint
    if 'val_stage_cfn-lint_passed' in df.columns:
        stage_stats.append({'Model': model, 'Stage': 'CFN-Lint', 'Pass Rate (%)': df['val_stage_cfn-lint_passed'].mean() * 100})
    # Security (Trivy as example)
    if 'val_stage_trivy_passed' in df.columns:
        stage_stats.append({'Model': model, 'Stage': 'Security (Trivy)', 'Pass Rate (%)': df['val_stage_trivy_passed'].mean() * 100})
    # Deployability
    if 'deploy_passed' in df.columns:
        stage_stats.append({'Model': model, 'Stage': 'Deployability', 'Pass Rate (%)': df['deploy_passed'].mean() * 100})

if stage_stats:
    stage_df = pd.DataFrame(stage_stats)
    plt.figure(figsize=(12, 6))
    sns.barplot(data=stage_df, x='Stage', y='Pass Rate (%)', hue='Model')
    plt.title('Validation Stage Pass Rates by Model')
    plt.ylim(0, 105)
    plt.legend(title="Model")
    plt.tight_layout()
    plt.show()
else:
    print("No stage pass rate data available.")

## 3. Terminal Failure Analysis
If a scenario ultimately fails (`final_validation_passed` == False), which stage was the bottleneck? We check stages in order (YAML -> CFN-Lint -> Security -> Deploy) to find the terminal failure.

In [ ]:
terminal_failures = []

for model, df in dataframes.items():
    failed_df = df[df['final_validation_passed'] == False]
    for _, row in failed_df.iterrows():
        terminal_stage = "Unknown"
        if 'val_stage_yaml_passed' in row and not row['val_stage_yaml_passed']:
            terminal_stage = "YAML"
        elif 'val_stage_cfn-lint_passed' in row and not row['val_stage_cfn-lint_passed']:
            terminal_stage = "CFN-Lint"
        elif 'val_stage_trivy_passed' in row and pd.notna(row['val_stage_trivy_passed']) and not row['val_stage_trivy_passed']:
            terminal_stage = "Security"
        elif 'deploy_passed' in row and not row['deploy_passed']:
            terminal_stage = "Deploy"
            
        terminal_failures.append({'Model': model, 'Terminal Stage': terminal_stage})

if terminal_failures:
    term_df = pd.DataFrame(terminal_failures)
    # Count occurrences
    term_counts = term_df.groupby(['Model', 'Terminal Stage']).size().reset_index(name='Count')
    
    plt.figure(figsize=(10, 6))
    sns.barplot(data=term_counts, x='Terminal Stage', y='Count', hue='Model')
    plt.title('Terminal Failures: Which Stage Caused the Final Failure?')
    plt.ylabel('Number of Scenarios Failed')
    plt.tight_layout()
    plt.show()
else:
    print("No terminal failures found or required columns missing.")

## 4. Error Analysis
Detailed breakdown of errors by specific validation and deployment stages.

### 4.1 Deployment Errors and Logs
Extracting and counting the most common deployment error messages and deployment log snippets.

In [ ]:
def plot_top_errors(df_dict, columns, title):
    all_errors = []
    for model, df in df_dict.items():
        for col in columns:
            if col in df.columns:
                errors = df[col].dropna().astype(str)
                errors = errors[~errors.isin(['None', 'nan', ''])]
                for err in errors:
                    if '|' in err:
                        all_errors.extend([f"[{model}] {e.strip()[:80]}" for e in err.split('|') if e.strip()])
                    else:
                        all_errors.append(f"[{model}] {err.strip()[:80]}")
                        
    if all_errors:
        counter = Counter(all_errors)
        top_errs = counter.most_common(10)
        err_df = pd.DataFrame(top_errs, columns=['Error Message', 'Frequency'])
        
        plt.figure(figsize=(10, 5))
        sns.barplot(data=err_df, y='Error Message', x='Frequency', color='coral')
        plt.title(title)
        plt.xlabel('Frequency')
        plt.ylabel('Snippet')
        plt.tight_layout()
        plt.show()
    else:
        print(f"No data found for: {title}")

plot_top_errors(dataframes, ['deploy_error_message', 'deploy_logs'], 'Top 10 Deployment Errors and Logs')

### 4.2 Deployment Failed Resources
Which specific CloudFormation resources are failing during LocalStack/AWS deployment?

In [ ]:
plot_top_errors(dataframes, ['deploy_failed_resources'], 'Top 10 Deploy Failed Resources')

### 4.3 Latest Error: YAML Validation
Most common YAML syntax or structural errors.

In [ ]:
plot_top_errors(dataframes, ['latest_error_yaml'], 'Top 10 YAML Validation Errors')

### 4.4 Latest Error: CFN-Lint Validation
Most common AWS CloudFormation Linter (cfn-lint) errors.

In [ ]:
plot_top_errors(dataframes, ['latest_error_cfn-lint'], 'Top 10 CFN-Lint Errors')